# Guided Scenario Runner

This notebook imports the shared simulation package, discovers every official case study, runs each A/B pair, and visualises the results.

The workflow stays aligned with the dashboard and API because every run comes from the same `case_study.json` manifests and preset library.


## Setup And Environment Check

This section prepares the notebook environment, locates the repository root, loads helper functions, and discovers the official case studies without requiring optional plotting libraries to be installed.

Install notebook-specific dependencies with `python3 -m pip install -r notebook/requirements.txt` before running the notebook. If you want to analyse a different scenario library, set `SIM_DATA_ROOT` before starting the kernel so the notebook and API read the same data root.

If you still see an older saved Plotly traceback under the setup cell, rerun that cell once to refresh the notebook output.


In [1]:
from __future__ import annotations

from collections import Counter
from html import escape
from importlib import import_module
from pathlib import Path
import sys

from IPython.display import HTML, Markdown, display


PERCENT_METRICS = {
    "service_level_within_15_min",
    "service_level_within_30_min",
    "table_utilization_overall",
    "reservation_fulfillment_rate",
    "average_table_fit_efficiency",
}
COUNT_METRICS = {
    "groups_served",
    "groups_abandoned",
    "max_queue_length",
}
METRIC_LABELS = {
    "average_wait_time": "Average Wait (min)",
    "max_wait_time": "Max Wait (min)",
    "p90_wait_time": "P90 Wait (min)",
    "max_queue_length": "Peak Queue (groups)",
    "groups_served": "Groups Served",
    "groups_abandoned": "Groups Abandoned",
    "service_level_within_15_min": "Service Level <=15 min (%)",
    "service_level_within_30_min": "Service Level <=30 min (%)",
    "table_utilization_overall": "Table Utilization (%)",
    "reservation_fulfillment_rate": "Reservation Fulfillment (%)",
    "average_reservation_delay": "Reservation Delay (min)",
    "average_table_fit_efficiency": "Table Fit Efficiency (%)",
}


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "restaurant_simulation").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from this notebook.")


def render_table(rows: list[dict[str, object]], columns: list[str]) -> HTML:
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body_rows = []
    for row in rows:
        body_cells = "".join(
            f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns
        )
        body_rows.append(f"<tr>{body_cells}</tr>")
    body = "".join(body_rows) or f"<tr><td colspan='{len(columns)}'>No rows</td></tr>"
    return HTML(
        "<table>"
        f"<thead><tr>{header}</tr></thead>"
        f"<tbody>{body}</tbody>"
        "</table>"
    )


def load_plotly():
    try:
        go_module = import_module("plotly.graph_objects")
        make_subplots_fn = import_module("plotly.subplots").make_subplots
        return go_module, make_subplots_fn, None
    except ModuleNotFoundError as error:
        return None, None, error


def metric_label(metric: str) -> str:
    return METRIC_LABELS.get(metric, metric.replace("_", " ").title())


def scale_metric(metric: str, value: float | int) -> float:
    if metric in PERCENT_METRICS:
        return float(value) * 100
    return float(value)


def format_metric_value(metric: str, value: float | int) -> str:
    scaled = scale_metric(metric, value)
    if metric in COUNT_METRICS:
        return f"{scaled:.0f}"
    if metric in PERCENT_METRICS:
        return f"{scaled:.1f}%"
    return f"{scaled:.1f}"


def format_delta_value(metric: str, value: float | int) -> str:
    scaled = scale_metric(metric, value)
    sign = "+" if scaled > 0 else ""
    if metric in COUNT_METRICS:
        return f"{sign}{scaled:.0f}"
    if metric in PERCENT_METRICS:
        return f"{sign}{scaled:.1f} pp"
    return f"{sign}{scaled:.1f} min"


def short_case_label(case) -> str:
    prefix = case.title.split(":", 1)[0].strip()
    return prefix or case.case_study


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CODE_ROOT = REPO_ROOT / "code"

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from restaurant_simulation import discover_case_studies, resolve_data_root, run_case_pair

DATA_ROOT = resolve_data_root()
case_studies = discover_case_studies(data_root=DATA_ROOT)
case_lookup = {case.case_study: case for case in case_studies}

print(f"Repository root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Discovered {len(case_studies)} official case studies.")
print("Use the selector cell below to switch between one pair and the full official set.")


Repository root: /Users/grantwright/Desktop/COMP 1110/COMP-1110-C08-add_scenario_run
Data root: /Users/grantwright/Desktop/COMP 1110/COMP-1110-C08-add_scenario_run/data
Discovered 7 official case studies.
Use the selector cell below to switch between one pair and the full official set.


## Scenario Selector

This section exposes the main switch for the notebook. Change the parameter in the next cell to either run one pair in detail or keep the full official scenario set.


In [2]:
AVAILABLE_CASE_STUDIES = [case.case_study for case in case_studies]

# Change this single parameter to switch the notebook scope.
# Use a specific pair id such as "pair_01_table_mix", or use "ALL".
SELECTED_CASE_STUDY = "ALL"

if SELECTED_CASE_STUDY == "ALL":
    selected_cases = case_studies
    selected_case_label = "All official case studies"
else:
    if SELECTED_CASE_STUDY not in AVAILABLE_CASE_STUDIES:
        raise ValueError(
            f"Unknown case study: {SELECTED_CASE_STUDY}. Available options: {', '.join(AVAILABLE_CASE_STUDIES)}"
        )
    selected_cases = [case_lookup[SELECTED_CASE_STUDY]]
    selected_case_label = case_lookup[SELECTED_CASE_STUDY].title

print("Available case studies:")
for case in case_studies:
    print(f"- {case.case_study}: {case.title}")
print(f"Current selection: {selected_case_label}")


Available case studies:
- pair_01_table_mix: Pair 01: Dessert Lounge vs Family Trattoria
- pair_02_single_vs_multi_queue: Pair 02: Single Queue vs Multi Queue
- pair_03_coarse_vs_fine_queue: Pair 03: Coarse vs Fine Queue Buckets
- pair_04_no_reservation_hold_vs_hold: Pair 04: No Reservation Hold vs Reservation Hold
- pair_05_low_vs_high_cleaning_capacity: Pair 05: Low vs High Cleaning Capacity
- pair_06_no_abandonment_vs_abandonment: Pair 06: No Abandonment vs Abandonment Enabled
- pair_07_fcfs_vs_best_fit: Pair 07: FCFS vs Best-Fit Seating
Current selection: All official case studies


## Scenario Execution

This section runs the selected official pair or the full official set, then materializes the main metrics and delta tables used by the rest of the notebook.


In [3]:
comparisons = [run_case_pair(case.case_study, data_root=DATA_ROOT) for case in selected_cases]

metrics_rows = []
delta_rows = []
for comparison in comparisons:
    case = case_lookup[comparison["case_study"]]
    for version in ["A", "B"]:
        metrics_rows.append(
            {
                "case_study": comparison["case_study"],
                "title": case.title,
                "version": version,
                **comparison[version]["metrics"],
            }
        )
    delta_rows.append(
        {
            "case_study": comparison["case_study"],
            "title": case.title,
            **comparison["metric_deltas_b_minus_a"],
        }
    )

print(f"Selection scope: {selected_case_label}")
print(f"Ran {len(comparisons)} pair comparison(s).")
print("Per-version metrics")
display(
    render_table(
        metrics_rows,
        [
            "case_study",
            "version",
            "average_wait_time",
            "groups_served",
            "groups_abandoned",
            "table_utilization_overall",
        ],
    )
)
print("Metric deltas (B minus A)")
display(
    render_table(
        delta_rows,
        [
            "case_study",
            "average_wait_time",
            "groups_served",
            "groups_abandoned",
            "table_utilization_overall",
        ],
    )
)


Selection scope: All official case studies
Ran 7 pair comparison(s).
Per-version metrics


case_study,version,average_wait_time,groups_served,groups_abandoned,table_utilization_overall
pair_01_table_mix,A,80.875,16,0,0.1529
pair_01_table_mix,B,18.0,16,0,0.2071
pair_02_single_vs_multi_queue,A,7.5625,16,0,0.1196
pair_02_single_vs_multi_queue,B,2.4375,16,0,0.1196
pair_03_coarse_vs_fine_queue,A,8.1875,16,0,0.2335
pair_03_coarse_vs_fine_queue,B,8.1875,16,0,0.2335
pair_04_no_reservation_hold_vs_hold,A,17.3333,15,0,0.2528
pair_04_no_reservation_hold_vs_hold,B,17.4667,15,0,0.2528
pair_05_low_vs_high_cleaning_capacity,A,9.2778,18,0,0.2578
pair_05_low_vs_high_cleaning_capacity,B,7.1111,18,0,0.2435


Metric deltas (B minus A)


case_study,average_wait_time,groups_served,groups_abandoned,table_utilization_overall
pair_01_table_mix,-62.875,0.0,0.0,0.0542
pair_02_single_vs_multi_queue,-5.125,0.0,0.0,0.0
pair_03_coarse_vs_fine_queue,0.0,0.0,0.0,0.0
pair_04_no_reservation_hold_vs_hold,0.1334,0.0,0.0,0.0
pair_05_low_vs_high_cleaning_capacity,-2.1667,0.0,0.0,-0.0143
pair_06_no_abandonment_vs_abandonment,-45.9342,-7.0,7.0,-0.0794
pair_07_fcfs_vs_best_fit,-1.2666,0.0,0.0,0.0


## Headline Metric Comparison

This section compares the most important service outcomes across the current selection. If Plotly is available, it also renders grouped charts; otherwise the summary table remains readable on its own.


In [4]:
summary_table_metrics = [
    "average_wait_time",
    "max_queue_length",
    "groups_served",
    "groups_abandoned",
    "service_level_within_15_min",
    "table_utilization_overall",
]
summary_chart_metrics = [
    "average_wait_time",
    "max_queue_length",
    "service_level_within_15_min",
    "table_utilization_overall",
]
summary_columns = ["pair", "scenario", "version", *[metric_label(metric) for metric in summary_table_metrics]]

summary_rows = [
    {
        "pair": short_case_label(case_lookup[row["case_study"]]),
        "scenario": row["title"],
        "version": row["version"],
        **{
            metric_label(metric): format_metric_value(metric, row[metric])
            for metric in summary_table_metrics
        },
    }
    for row in metrics_rows
]

display(render_table(summary_rows, summary_columns))

go, make_subplots, plotly_error = load_plotly()
if go is None:
    print(f"Plotly is not available in this kernel ({plotly_error}). The summary table above still contains the core comparison results.")
else:
    summary_chart = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=tuple(metric_label(metric) for metric in summary_chart_metrics),
    )

    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
    case_labels = [short_case_label(case) for case in selected_cases]
    for metric, (row_idx, col_idx) in zip(summary_chart_metrics, positions):
        values_a = []
        values_b = []
        for case in selected_cases:
            option_a = next(
                row for row in metrics_rows if row["case_study"] == case.case_study and row["version"] == "A"
            )
            option_b = next(
                row for row in metrics_rows if row["case_study"] == case.case_study and row["version"] == "B"
            )
            values_a.append(scale_metric(metric, option_a[metric]))
            values_b.append(scale_metric(metric, option_b[metric]))
        summary_chart.add_trace(
            go.Bar(
                name="Option A",
                x=case_labels,
                y=values_a,
                legendgroup="A",
                showlegend=metric == summary_chart_metrics[0],
            ),
            row=row_idx,
            col=col_idx,
        )
        summary_chart.add_trace(
            go.Bar(
                name="Option B",
                x=case_labels,
                y=values_b,
                legendgroup="B",
                showlegend=metric == summary_chart_metrics[0],
            ),
            row=row_idx,
            col=col_idx,
        )
        summary_chart.update_yaxes(title_text=metric_label(metric), row=row_idx, col=col_idx)

    summary_chart.update_layout(
        height=820,
        width=1180,
        barmode="group",
        title=f"Headline metrics for {selected_case_label}",
    )
    summary_chart.show()

largest_wait_shift = max(delta_rows, key=lambda row: abs(row["average_wait_time"]))
wait_direction = "reduced" if largest_wait_shift["average_wait_time"] < 0 else "increased"
best_service_row = max(metrics_rows, key=lambda row: row["service_level_within_15_min"])
highest_util_row = max(metrics_rows, key=lambda row: row["table_utilization_overall"])

display(
    Markdown(
        "### Headline takeaways\n"
        f"- The largest wait-time shift appears in **{largest_wait_shift['title']}**, where Option B **{wait_direction}** average wait by **{format_delta_value('average_wait_time', largest_wait_shift['average_wait_time'])}** relative to Option A.\n"
        f"- The strongest 15-minute service level in the current selection is **{best_service_row['title']} ({best_service_row['version']})** at **{format_metric_value('service_level_within_15_min', best_service_row['service_level_within_15_min'])}**.\n"
        f"- The highest overall table utilization in the current selection is **{highest_util_row['title']} ({highest_util_row['version']})** at **{format_metric_value('table_utilization_overall', highest_util_row['table_utilization_overall'])}**."
    )
)


pair,scenario,version,Average Wait (min),Peak Queue (groups),Groups Served,Groups Abandoned,Service Level <=15 min (%),Table Utilization (%)
Pair 01,Pair 01: Dessert Lounge vs Family Trattoria,A,80.9,7,16,0,56.2%,15.3%
Pair 01,Pair 01: Dessert Lounge vs Family Trattoria,B,18.0,5,16,0,68.8%,20.7%
Pair 02,Pair 02: Single Queue vs Multi Queue,A,7.6,5,16,0,75.0%,12.0%
Pair 02,Pair 02: Single Queue vs Multi Queue,B,2.4,2,16,0,93.8%,12.0%
Pair 03,Pair 03: Coarse vs Fine Queue Buckets,A,8.2,4,16,0,81.2%,23.4%
Pair 03,Pair 03: Coarse vs Fine Queue Buckets,B,8.2,4,16,0,81.2%,23.4%
Pair 04,Pair 04: No Reservation Hold vs Reservation Hold,A,17.3,5,15,0,53.3%,25.3%
Pair 04,Pair 04: No Reservation Hold vs Reservation Hold,B,17.5,5,15,0,53.3%,25.3%
Pair 05,Pair 05: Low vs High Cleaning Capacity,A,9.3,4,18,0,72.2%,25.8%
Pair 05,Pair 05: Low vs High Cleaning Capacity,B,7.1,3,18,0,83.3%,24.3%


### Headline takeaways
- The largest wait-time shift appears in **Pair 01: Dessert Lounge vs Family Trattoria**, where Option B **reduced** average wait by **-62.9 min** relative to Option A.
- The strongest 15-minute service level in the current selection is **Pair 02: Single Queue vs Multi Queue (B)** at **93.8%**.
- The highest overall table utilization in the current selection is **Pair 05: Low vs High Cleaning Capacity (A)** at **25.8%**.

## Delta Analysis

This section focuses on the change from Option A to Option B, helping the reader understand what actually improves, worsens, or stays flat after switching policies.


In [5]:
interesting_delta_metrics = [
    "average_wait_time",
    "max_queue_length",
    "service_level_within_15_min",
    "table_utilization_overall",
]
delta_columns = ["pair", "scenario", *[metric_label(metric) for metric in interesting_delta_metrics]]

delta_rows_for_display = [
    {
        "pair": short_case_label(case_lookup[row["case_study"]]),
        "scenario": row["title"],
        **{
            metric_label(metric): format_delta_value(metric, row[metric])
            for metric in interesting_delta_metrics
        },
    }
    for row in delta_rows
]

display(render_table(delta_rows_for_display, delta_columns))

go, make_subplots, plotly_error = load_plotly()
if go is None:
    print(f"Plotly is not available in this kernel ({plotly_error}). The delta table above still shows the B minus A changes.")
else:
    delta_chart = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=tuple(metric_label(metric) for metric in interesting_delta_metrics),
    )
    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
    case_labels = [short_case_label(case_lookup[row["case_study"]]) for row in delta_rows]
    for metric, (row_idx, col_idx) in zip(interesting_delta_metrics, positions):
        delta_chart.add_trace(
            go.Bar(
                name=metric_label(metric),
                x=case_labels,
                y=[scale_metric(metric, row[metric]) for row in delta_rows],
            ),
            row=row_idx,
            col=col_idx,
        )
        delta_chart.add_hline(y=0, row=row_idx, col=col_idx)
        delta_chart.update_yaxes(title_text=metric_label(metric), row=row_idx, col=col_idx)
    delta_chart.update_layout(
        title=f"Option B minus Option A for {selected_case_label}",
        height=820,
        width=1180,
        showlegend=False,
    )
    delta_chart.show()

largest_wait_gain = min(delta_rows, key=lambda row: row["average_wait_time"])
largest_service_gain = max(delta_rows, key=lambda row: row["service_level_within_15_min"])
largest_util_shift = max(delta_rows, key=lambda row: abs(row["table_utilization_overall"]))
util_direction = "increased" if largest_util_shift["table_utilization_overall"] > 0 else "decreased"

display(
    Markdown(
        "### Delta reading guide\n"
        "- For **Average Wait** and **Peak Queue**, a negative delta means Option B relieved congestion relative to Option A.\n"
        "- For **Service Level** and **Table Utilization**, a positive delta means Option B improved the outcome.\n"
        f"- The biggest wait reduction in the current selection is **{largest_wait_gain['title']}**, with a change of **{format_delta_value('average_wait_time', largest_wait_gain['average_wait_time'])}**.\n"
        f"- The strongest service-level gain appears in **{largest_service_gain['title']}**, at **{format_delta_value('service_level_within_15_min', largest_service_gain['service_level_within_15_min'])}**.\n"
        f"- The largest utilization shift appears in **{largest_util_shift['title']}**, where Option B **{util_direction}** utilization by **{format_delta_value('table_utilization_overall', largest_util_shift['table_utilization_overall'])}**."
    )
)


pair,scenario,Average Wait (min),Peak Queue (groups),Service Level <=15 min (%),Table Utilization (%)
Pair 01,Pair 01: Dessert Lounge vs Family Trattoria,-62.9 min,-2,+12.5 pp,+5.4 pp
Pair 02,Pair 02: Single Queue vs Multi Queue,-5.1 min,-3,+18.8 pp,0.0 pp
Pair 03,Pair 03: Coarse vs Fine Queue Buckets,0.0 min,0,0.0 pp,0.0 pp
Pair 04,Pair 04: No Reservation Hold vs Reservation Hold,+0.1 min,0,0.0 pp,0.0 pp
Pair 05,Pair 05: Low vs High Cleaning Capacity,-2.2 min,-1,+11.1 pp,-1.4 pp
Pair 06,Pair 06: No Abandonment vs Abandonment Enabled,-45.9 min,-4,+10.5 pp,-7.9 pp
Pair 07,Pair 07: FCFS vs Best-Fit Seating,-1.3 min,0,-6.7 pp,0.0 pp


### Delta reading guide
- For **Average Wait** and **Peak Queue**, a negative delta means Option B relieved congestion relative to Option A.
- For **Service Level** and **Table Utilization**, a positive delta means Option B improved the outcome.
- The biggest wait reduction in the current selection is **Pair 01: Dessert Lounge vs Family Trattoria**, with a change of **-62.9 min**.
- The strongest service-level gain appears in **Pair 02: Single Queue vs Multi Queue**, at **+18.8 pp**.
- The largest utilization shift appears in **Pair 06: No Abandonment vs Abandonment Enabled**, where Option B **decreased** utilization by **-7.9 pp**.

## Pair Deep Dive

This section drills into each selected pair with per-case metric tables, outcome breakdowns, and queue summaries, plus charts when optional plotting support is available.


In [6]:
detail_metrics = [
    "average_wait_time",
    "max_queue_length",
    "service_level_within_15_min",
    "table_utilization_overall",
    "average_table_fit_efficiency",
]
chart_metrics = [
    "average_wait_time",
    "max_queue_length",
    "service_level_within_15_min",
    "table_utilization_overall",
]

go, make_subplots, plotly_error = load_plotly()

for comparison in comparisons:
    case = case_lookup[comparison["case_study"]]

    metric_rows = [
        {
            "metric": metric_label(metric),
            "Option A": format_metric_value(metric, comparison["A"]["metrics"][metric]),
            "Option B": format_metric_value(metric, comparison["B"]["metrics"][metric]),
            "B - A": format_delta_value(metric, comparison["metric_deltas_b_minus_a"][metric]),
        }
        for metric in detail_metrics
    ]

    outcome_counts = {
        version: Counter(outcome["status"] for outcome in comparison[version]["group_outcomes"])
        for version in ["A", "B"]
    }
    outcome_statuses = sorted({status for counter in outcome_counts.values() for status in counter})
    outcome_rows = [
        {
            "version": version,
            **{status: outcome_counts[version].get(status, 0) for status in outcome_statuses},
        }
        for version in ["A", "B"]
    ]

    queue_rows = [
        {
            "version": version,
            "minute": snapshot["minute"],
            "clock": snapshot["clock"],
            "total_waiting": snapshot["total_waiting"],
        }
        for version in ["A", "B"]
        for snapshot in comparison[version]["queue_snapshots"]
    ]
    queue_summary_rows = []
    for version in ["A", "B"]:
        version_rows = [row for row in queue_rows if row["version"] == version]
        if version_rows:
            peak_row = max(version_rows, key=lambda row: row["total_waiting"])
        else:
            peak_row = {"clock": "-", "total_waiting": 0}
        queue_summary_rows.append(
            {
                "version": version,
                "peak_waiting": peak_row["total_waiting"],
                "peak_time": peak_row["clock"],
                "final_waiting": version_rows[-1]["total_waiting"] if version_rows else 0,
                "snapshots": len(version_rows),
            }
        )

    seating_event_rows = []
    for version in ["A", "B"]:
        seat_events = [entry for entry in comparison[version]["event_log"] if entry["event_type"] == "seated"][:4]
        for event in seat_events:
            seating_event_rows.append(
                {
                    "version": version,
                    "time": event["clock"],
                    "group": event["group_id"],
                    "table": event["table_id"],
                    "message": event["message"],
                }
            )
    if not seating_event_rows:
        seating_event_rows = [
            {
                "version": "-",
                "time": "-",
                "group": "-",
                "table": "-",
                "message": "No seating events were recorded for this comparison.",
            }
        ]

    wait_delta = comparison["metric_deltas_b_minus_a"]["average_wait_time"]
    queue_delta = comparison["metric_deltas_b_minus_a"]["max_queue_length"]
    service_delta = comparison["metric_deltas_b_minus_a"]["service_level_within_15_min"]
    util_delta = comparison["metric_deltas_b_minus_a"]["table_utilization_overall"]
    if wait_delta < 0 and util_delta >= 0:
        operational_reading = "Option B improves guest flow without sacrificing space usage, so it would usually be the stronger operating rule if staff can follow it consistently."
    elif wait_delta < 0:
        operational_reading = "Option B reduces congestion, but the gain comes with lower space usage, so managers would need to balance speed against throughput."
    elif wait_delta > 0 and util_delta > 0:
        operational_reading = "Option B uses capacity more aggressively, but the extra utilization comes with slower seating, which may only be acceptable in venues that prioritize occupancy over responsiveness."
    else:
        operational_reading = "This pair stays fairly stable on the core metrics, so any preference for A or B is likely to come from operational simplicity, staff workload, or edge-case behavior."

    print(case.title)
    display(render_table(metric_rows, ["metric", "Option A", "Option B", "B - A"]))
    display(render_table(outcome_rows, ["version", *outcome_statuses]))
    display(render_table(queue_summary_rows, ["version", "peak_waiting", "peak_time", "final_waiting", "snapshots"]))
    display(render_table(seating_event_rows, ["version", "time", "group", "table", "message"]))
    display(
        Markdown(
            "#### Pair takeaway\n"
            f"- Focus factor: **{case.focus_label or 'Operational trade-off'}**.\n"
            f"- Option B changed average wait by **{format_delta_value('average_wait_time', wait_delta)}**, peak queue by **{format_delta_value('max_queue_length', queue_delta)}**, and 15-minute service level by **{format_delta_value('service_level_within_15_min', service_delta)}**.\n"
            f"- Operational reading: {operational_reading}\n"
            "- Real-world note: restaurants would still weigh host training cost, perceived fairness, special requests, and table-sharing exceptions that are not fully modeled here."
        )
    )

    if go is None:
        print(f"Plotly is not available in this kernel ({plotly_error}). The tables above provide the deep-dive summary for this pair.")
        continue

    subplot = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(
            "Core metrics",
            "Group outcomes",
            "Queue pressure over time",
        ),
    )

    subplot.add_trace(
        go.Bar(
            name="Option A",
            x=[metric_label(metric) for metric in chart_metrics],
            y=[scale_metric(metric, comparison["A"]["metrics"][metric]) for metric in chart_metrics],
        ),
        row=1,
        col=1,
    )
    subplot.add_trace(
        go.Bar(
            name="Option B",
            x=[metric_label(metric) for metric in chart_metrics],
            y=[scale_metric(metric, comparison["B"]["metrics"][metric]) for metric in chart_metrics],
        ),
        row=1,
        col=1,
    )

    for status in outcome_statuses:
        subplot.add_trace(
            go.Bar(
                name=f"status: {status}",
                x=["A", "B"],
                y=[outcome_counts["A"].get(status, 0), outcome_counts["B"].get(status, 0)],
            ),
            row=1,
            col=2,
        )

    for version in ["A", "B"]:
        version_rows = [row for row in queue_rows if row["version"] == version]
        subplot.add_trace(
            go.Scatter(
                x=[row["clock"] for row in version_rows],
                y=[row["total_waiting"] for row in version_rows],
                mode="lines+markers",
                name=f"Queue {version}",
            ),
            row=1,
            col=3,
        )

    subplot.update_layout(
        title=case.title,
        barmode="group",
        width=1380,
        height=480,
    )
    subplot.update_xaxes(title_text="Metric", row=1, col=1)
    subplot.update_xaxes(title_text="Version", row=1, col=2)
    subplot.update_xaxes(title_text="Simulation clock", row=1, col=3)
    subplot.update_yaxes(title_text="Scaled value", row=1, col=1)
    subplot.update_yaxes(title_text="Groups", row=1, col=2)
    subplot.update_yaxes(title_text="Waiting groups", row=1, col=3)
    subplot.show()


Pair 01: Dessert Lounge vs Family Trattoria


metric,Option A,Option B,B - A
Average Wait (min),80.9,18.0,-62.9 min
Peak Queue (groups),7,5,-2
Service Level <=15 min (%),56.2%,68.8%,+12.5 pp
Table Utilization (%),15.3%,20.7%,+5.4 pp
Table Fit Efficiency (%),91.9%,93.2%,+1.3 pp


version,completed
A,16
B,16


version,peak_waiting,peak_time,final_waiting,snapshots
A,6,19:18,0,46
B,4,19:18,0,46


version,time,group,table,message
A,17:40,H001,D01,Group H001 was seated at table D01.
A,17:45,H002,D09,Group H002 was seated at table D09.
A,17:52,H003,D10,Group H003 was seated at table D10.
A,18:08,H005,D05,Group H005 was seated at table D05.
B,17:40,H001,F01,Group H001 was seated at table F01.
B,17:45,H002,F07,Group H002 was seated at table F07.
B,17:52,H003,F08,Group H003 was seated at table F08.
B,18:00,H004,F09,Group H004 was seated at table F09.


...

metric,Option A,Option B,B - A
Average Wait (min),4.5,3.3,-1.3 min
Peak Queue (groups),2,2,0
Service Level <=15 min (%),93.3%,86.7%,-6.7 pp
Table Utilization (%),17.7%,17.7%,0.0 pp
Table Fit Efficiency (%),88.9%,90.8%,+1.9 pp


version,completed,no_show
A,15,1
B,15,1


version,peak_waiting,peak_time,final_waiting,snapshots
A,2,19:05,0,44
B,2,19:05,0,45


version,time,group,table,message
A,17:30,R001,G01,Group R001 was seated at table G01.
A,17:36,R002,G05,Group R002 was seated at table G05.
A,17:42,R003,G06,Group R003 was seated at table G06.
A,17:50,R004,G02,Group R004 was seated at table G02.
B,17:30,R001,G01,Group R001 was seated at table G01.
B,17:36,R002,G05,Group R002 was seated at table G05.
B,17:42,R003,G06,Group R003 was seated at table G06.
B,17:50,R004,G02,Group R004 was seated at table G02.


#### Pair takeaway
- Focus factor: **Seating heuristic**.
- Option B changed average wait by **-1.3 min**, peak queue by **0**, and 15-minute service level by **-6.7 pp**.
- Operational reading: Option B improves guest flow without sacrificing space usage, so it would usually be the stronger operating rule if staff can follow it consistently.
- Real-world note: restaurants would still weigh host training cost, perceived fairness, special requests, and table-sharing exceptions that are not fully modeled here.